# LeetCode 3131 — Find the Integer Added to Array I bằng Python

Link đề: https://leetcode.com/problems/find-the-integer-added-to-array-i/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def addedInteger(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mọi phần tử của `nums1` được **cộng thêm cùng một số nguyên `x`** (có thể âm hoặc bằng `0`) thì thu được mảng **bằng** `nums2` (hai mảng bằng nhau khi có **cùng các số với cùng số lần xuất hiện**, **không cần cùng thứ tự**); hãy tìm `x`.
2. Nắm các công cụ Python: `min`, `max`, `sum`, `sorted`, `Counter` (so sánh "đa tập" — multiset), phép chia nguyên `//` và vì sao dùng `//` thay `/`.
3. Đi từ cách **thử mọi giá trị `x` có thể** (vét cạn) → **thử `x` bằng cách ghép phần tử đầu của `nums1` với từng phần tử của `nums2`** → **hiệu của hai phần tử nhỏ nhất** → **hiệu của hai phần tử lớn nhất** → **hiệu hai tổng chia cho `n`**.
4. Hiểu mấu chốt: cộng cùng một `x` vào mọi phần tử **giữ nguyên thứ tự tương đối**, nên **phần tử nhỏ nhất** của `nums1` biến thành phần tử nhỏ nhất của `nums2`: `x = min(nums2) − min(nums1)`; tương tự cho `max` và cho **tổng**.
5. Tránh các lỗi kinh điển: lấy `nums2[0] − nums1[0]` **mà không sắp xếp** (các mảng không cùng thứ tự), nhầm dấu (`nums1 − nums2`), dùng `/` (ra số thực) khi chia tổng cho `n`, và quên rằng `x` có thể **âm** hay `0`.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1` và `nums2` **có cùng độ dài**.

Mỗi phần tử của `nums1` được **cộng thêm một số nguyên `x`** (cùng một `x` cho mọi phần tử; `x` có thể dương, âm hoặc bằng `0`). Sau khi cộng, `nums1` trở nên **bằng** `nums2`. Hai mảng gọi là **bằng nhau** nếu chúng chứa **cùng các số nguyên với cùng tần suất** (thứ tự **không** quan trọng).

Hãy trả về số nguyên `x`.

**Ví dụ 1**

- Input: `nums1 = [2, 6, 4]`, `nums2 = [9, 7, 5]`
- Output: `3`
- Giải thích: cộng `3` vào mỗi phần tử của `nums1`: `[5, 9, 7]`, bằng `nums2` (cùng các số `5, 7, 9`).

**Ví dụ 2**

- Input: `nums1 = [10]`, `nums2 = [5]`
- Output: `-5`
- Giải thích: `10 + (−5) = 5`.

**Ví dụ 3**

- Input: `nums1 = [1, 1, 1, 1]`, `nums2 = [1, 1, 1, 1]`
- Output: `0`
- Giải thích: hai mảng đã bằng nhau, `x = 0`.

**Ràng buộc:** `1 <= nums1.length == nums2.length <= 100`; `0 <= nums1[i], nums2[i] <= 1000`; dữ liệu được tạo sao cho **tồn tại** số nguyên `x` thỏa đề.

In [1]:
from typing import List
from collections import Counter
import random

examples = [
    ([2, 6, 4], [9, 7, 5], 3),
    ([10], [5], -5),
    ([1, 1, 1, 1], [1, 1, 1, 1], 0),
]
for nums1, nums2, expected in examples:
    print("nums1 =", nums1, "nums2 =", nums2, "-> kỳ vọng", expected)

nums1 = [2, 6, 4] nums2 = [9, 7, 5] -> kỳ vọng 3
nums1 = [10] nums2 = [5] -> kỳ vọng -5
nums1 = [1, 1, 1, 1] nums2 = [1, 1, 1, 1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. "Bằng nhau" kiểu đa tập: sắp xếp hoặc `Counter`

Hai mảng **bằng nhau không cần thứ tự** nếu sau khi **sắp xếp** chúng giống hệt nhau, hoặc nếu `Counter` (bộ đếm số lần xuất hiện) của chúng bằng nhau. `[5, 9, 7]` và `[9, 7, 5]` có cùng các số nên bằng nhau theo nghĩa của đề, dù `[5, 9, 7] != [9, 7, 5]` khi so sánh list trực tiếp.

In [2]:
a, b = [5, 9, 7], [9, 7, 5]
print(a == b)                                  # False: so list có tính thứ tự
print(sorted(a) == sorted(b))                  # True
print(Counter(a) == Counter(b))                # True
print(Counter([1, 1, 2]) == Counter([1, 2, 2]))   # False (khác tần suất)

False
True
True
False


### 3.2. Cộng một số vào mọi phần tử

`[v + x for v in nums]` tạo list mới với mỗi phần tử cộng thêm `x`. Phép cộng cùng một số vào mọi phần tử **giữ nguyên thứ tự**: nếu `a < b` thì `a + x < b + x`. Do đó **phần tử nhỏ nhất vẫn là nhỏ nhất**, phần tử lớn nhất vẫn là lớn nhất.

In [3]:
nums1, x = [2, 6, 4], 3
shifted = [v + x for v in nums1]
print(shifted)                                  # [5, 9, 7]
print(min(nums1), min(shifted), max(nums1), max(shifted))     # 2 5 6 9

[5, 9, 7]
2 5 6 9


### 3.3. `min`, `max`, `sum`

- `min(a)`, `max(a)`: phần tử nhỏ nhất / lớn nhất của `a`.
- `sum(a)`: tổng các phần tử.

Khi cộng `x` vào `n` phần tử thì **tổng tăng đúng `n · x`**: `sum(nums1 + x) = sum(nums1) + n · x`.

In [4]:
nums1 = [2, 6, 4]
nums2 = [9, 7, 5]
print(sum(nums1), sum(nums2))                       # 12 21
print(sum(nums2) - sum(nums1), len(nums1) * 3)      # 9 9: hiệu tổng = n * x

12 21
9 9


### 3.4. Chia nguyên `//` và chia thực `/`

`9 / 3` cho `3.0` (số thực). `9 // 3` cho `3` (số nguyên). Vì `x` phải là **số nguyên**, ta dùng `//`. Cẩn thận với số âm: `-10 // 2 = -5` (đúng), nhưng `-7 // 2 = -4` (làm tròn **xuống**). Trong bài này phép chia luôn chính xác (hiệu tổng chia hết cho `n`), nên không phải lo làm tròn.

In [5]:
print(9 / 3, 9 // 3)           # 3.0 3
print(-10 // 2)                # -5
print(-7 // 2, -7 / 2)         # -4 -3.5   (cẩn thận: làm tròn xuống)

3.0 3
-5
-4 -3.5


### 3.5. `range` với bước âm và giá trị `x` có thể âm

`x` có thể từ `-1000` (nums1 = 1000, nums2 = 0) đến `1000` (nums1 = 0, nums2 = 1000). `range(-1000, 1001)` duyệt mọi giá trị đó; `x` âm hoàn toàn bình thường.

In [6]:
print(len(range(-1000, 1001)))          # 2001 giá trị
print(list(range(-2, 3)))               # [-2, -1, 0, 1, 2]

2001
[-2, -1, 0, 1, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums1 = [2, 6, 4]`, `nums2 = [9, 7, 5]`.

Sắp xếp cả hai: `nums1 → [2, 4, 6]`, `nums2 → [5, 7, 9]`. Cộng `x` vào mọi phần tử của `nums1` giữ nguyên thứ tự nên phần tử thứ `i` của `nums1` đã sắp xếp ứng với phần tử thứ `i` của `nums2` đã sắp xếp:

| vị trí (sau sắp xếp) | `nums1` | `nums2` | hiệu |
|---|---|---|---|
| 0 | 2 | 5 | 3 |
| 1 | 4 | 7 | 3 |
| 2 | 6 | 9 | 3 |

Mọi hiệu đều bằng `3`, đó chính là `x`. Chỉ cần **một** hiệu là đủ (vì đề đảm bảo `x` tồn tại): ví dụ hiệu của hai phần tử nhỏ nhất `5 − 2 = 3`.

Chú ý: ghép **theo thứ tự ban đầu** (`nums2[0] − nums1[0] = 9 − 2 = 7`) là **sai**, vì hai mảng không nhất thiết cùng thứ tự.

In [7]:
nums1, nums2 = [2, 6, 4], [9, 7, 5]
s1, s2 = sorted(nums1), sorted(nums2)
print(s1, s2)
print([b - a for a, b in zip(s1, s2)])               # [3, 3, 3]
print("ghép theo thứ tự ban đầu (SAI):", [b - a for a, b in zip(nums1, nums2)])     # [7, 1, 1]

[2, 4, 6] [5, 7, 9]
[3, 3, 3]
ghép theo thứ tự ban đầu (SAI): [7, 1, 1]


## 5. Cách 1 — Brute Force / baseline: thử mọi giá trị `x`

**Ý tưởng.** Vì `0 <= nums[i] <= 1000`, `x` chỉ có thể nằm trong `[-1000, 1000]` (`2001` giá trị). Với mỗi `x`: cộng `x` vào `nums1`, so với `nums2` bằng `Counter` (bỏ qua thứ tự). `x` đầu tiên khớp là đáp án.

In [8]:
def added_integer_bruteforce(nums1: List[int], nums2: List[int]) -> int:
    target = Counter(nums2)
    for x in range(-1000, 1001):                         # mọi giá trị x có thể
        if Counter(v + x for v in nums1) == target:      # nums1 + x có bằng nums2 (kiểu đa tập)?
            return x
    return 0                                             # theo đề luôn có đáp án


for nums1, nums2, expected in examples:
    print(added_integer_bruteforce(nums1, nums2), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
-5 (kỳ vọng -5 )
0 (kỳ vọng 0 )


**Độ phức tạp (Cách 1).** Gọi `n` là độ dài, `V = 2001` là số giá trị `x`. Mỗi `x` tạo `Counter` `O(n)` ⇒ **Time `O(V · n)`**, **Space `O(n)`**. Với `n <= 100`: khoảng `2 · 10^5` phép, vẫn nhanh.

**Nút thắt:** ta thử cả những `x` rõ ràng không thể (ví dụ `x = 500` trong khi mọi `nums2` nhỏ hơn). Ta có thể rút ra `x` trực tiếp từ dữ liệu.

## 6. Ý tưởng thuật toán / bước đột phá

Vì `nums1 + x` **bằng** `nums2` (cùng đa tập), nên:

- **Phần tử nhỏ nhất** của `nums1 + x` là `min(nums1) + x`, và đó chính là `min(nums2)`. Suy ra `x = min(nums2) − min(nums1)`.
- Tương tự `x = max(nums2) − max(nums1)`.
- Tổng: `sum(nums2) = sum(nums1) + n · x` ⇒ `x = (sum(nums2) − sum(nums1)) / n`.

Cả ba cho cùng `x` (vì đề đảm bảo `x` tồn tại). Thời gian `O(n)` cho mỗi công thức, **không thử nhiều `x`**.

In [9]:
for nums1, nums2, expected in examples:
    by_min = min(nums2) - min(nums1)
    by_max = max(nums2) - max(nums1)
    by_sum = (sum(nums2) - sum(nums1)) // len(nums1)
    print(f"min: {by_min}, max: {by_max}, sum: {by_sum} (kỳ vọng {expected})")
    assert by_min == by_max == by_sum == expected

min: 3, max: 3, sum: 3 (kỳ vọng 3)
min: -5, max: -5, sum: -5 (kỳ vọng -5)
min: 0, max: 0, sum: 0 (kỳ vọng 0)


## 7. Các cách tiếp theo

### Cách 2 — Ghép phần tử đầu của `nums1` với từng phần tử của `nums2`

**Giải thích.** `nums1[0] + x` phải bằng **một** phần tử nào đó của `nums2`. Vậy `x` là một trong các `nums2[j] − nums1[0]`. Với mỗi ứng viên, kiểm tra có `nums1 + x == nums2` (đa tập) không. Tối đa `n` ứng viên (thay vì `2001`).

In [10]:
def added_integer_candidates(nums1: List[int], nums2: List[int]) -> int:
    target = Counter(nums2)
    for y in nums2:
        x = y - nums1[0]                                  # ứng viên: nums1[0] + x = y
        if Counter(v + x for v in nums1) == target:       # kiểm tra mọi phần tử
            return x
    return 0


for nums1, nums2, expected in examples:
    print(added_integer_candidates(nums1, nums2), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
-5 (kỳ vọng -5 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 2:** `n` ứng viên × kiểm tra `O(n)` ⇒ **Time `O(n²)`**, **Space `O(n)`**.

### Cách 3 — Hiệu hai phần tử nhỏ nhất (cách nộp)

**Giải thích.** `x = min(nums2) − min(nums1)`. Một dòng, `O(n)`. Lưu ý đây dựa vào đảm bảo của đề rằng `x` tồn tại; nếu đề không đảm bảo ta phải kiểm tra thêm.

In [11]:
def added_integer_min(nums1: List[int], nums2: List[int]) -> int:
    return min(nums2) - min(nums1)


for nums1, nums2, expected in examples:
    print(added_integer_min(nums1, nums2), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
-5 (kỳ vọng -5 )
0 (kỳ vọng 0 )


In [12]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionMin:
    def addedInteger(self, nums1: List[int], nums2: List[int]) -> int:
        return min(nums2) - min(nums1)


print(SolutionMin().addedInteger([2, 6, 4], [9, 7, 5]))      # 3

3


**Độ phức tạp Cách 3:** **Time `O(n)`** (hai lần duyệt tìm `min`), **Space `O(1)`**.

### Cách 4 — Hiệu hai phần tử lớn nhất

**Giải thích.** Tương tự: `x = max(nums2) − max(nums1)`. Cho cùng kết quả vì cộng `x` giữ nguyên thứ tự.

In [13]:
def added_integer_max(nums1: List[int], nums2: List[int]) -> int:
    return max(nums2) - max(nums1)


for nums1, nums2, expected in examples:
    print(added_integer_max(nums1, nums2), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
-5 (kỳ vọng -5 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 4:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 5 — Hiệu hai tổng chia cho `n`

**Giải thích.** `sum(nums2) − sum(nums1) = n · x` ⇒ `x = (sum(nums2) − sum(nums1)) // n`. Dùng `//` vì kết quả phải là số nguyên (và phép chia luôn chính xác do đề đảm bảo). Cách này không dùng `min/max` và "nhìn" mọi phần tử.

In [14]:
def added_integer_sum(nums1: List[int], nums2: List[int]) -> int:
    return (sum(nums2) - sum(nums1)) // len(nums1)


for nums1, nums2, expected in examples:
    print(added_integer_sum(nums1, nums2), "(kỳ vọng", expected, ")")

3 (kỳ vọng 3 )
-5 (kỳ vọng -5 )
0 (kỳ vọng 0 )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`**.

## 8. TRACE từng bước

Ta trace Cách 2 (ứng viên) và Cách 1 (một vài giá trị `x`) trên `nums1 = [2, 6, 4]`, `nums2 = [9, 7, 5]`.

In [15]:
def trace_candidates(nums1: List[int], nums2: List[int]) -> int:
    target = Counter(nums2)
    print("nums1 =", nums1, "| nums2 =", nums2, "| nums1[0] =", nums1[0])
    for y in nums2:
        x = y - nums1[0]
        shifted = [v + x for v in nums1]
        ok = Counter(shifted) == target
        print(f"ứng viên y={y}: x = {y} - {nums1[0]} = {x} -> nums1 + x = {shifted} -> {'KHỚP' if ok else 'không khớp'}")
        if ok:
            return x
    return 0


assert trace_candidates([2, 6, 4], [9, 7, 5]) == 3

nums1 = [2, 6, 4] | nums2 = [9, 7, 5] | nums1[0] = 2
ứng viên y=9: x = 9 - 2 = 7 -> nums1 + x = [9, 13, 11] -> không khớp
ứng viên y=7: x = 7 - 2 = 5 -> nums1 + x = [7, 11, 9] -> không khớp
ứng viên y=5: x = 5 - 2 = 3 -> nums1 + x = [5, 9, 7] -> KHỚP


In [16]:
def trace_formulas(nums1: List[int], nums2: List[int]) -> int:
    n = len(nums1)
    print(f"min: {min(nums2)} - {min(nums1)} = {min(nums2) - min(nums1)}")
    print(f"max: {max(nums2)} - {max(nums1)} = {max(nums2) - max(nums1)}")
    print(f"sum: ({sum(nums2)} - {sum(nums1)}) // {n} = {(sum(nums2) - sum(nums1)) // n}")
    return min(nums2) - min(nums1)


assert trace_formulas([2, 6, 4], [9, 7, 5]) == 3

min: 5 - 2 = 3
max: 9 - 6 = 3
sum: (21 - 12) // 3 = 3


Bảng trace Cách 2 với `[2, 6, 4]` và `[9, 7, 5]`:

| ứng viên `y` | `x = y − 2` | `nums1 + x` | khớp `nums2`? |
|---|---|---|---|
| 9 | 7 | `[9, 13, 11]` | không |
| 7 | 5 | `[7, 11, 9]` | không |
| 5 | 3 | `[5, 9, 7]` | **có** ⇒ `x = 3` |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (`min(nums2) − min(nums1)`): một dòng, `O(n)` thời gian, `O(1)` bộ nhớ.

In [17]:
from typing import List

class Solution:
    def addedInteger(self, nums1: List[int], nums2: List[int]) -> int:
        # cộng cùng x không đổi thứ tự: phần tử nhỏ nhất của nums1 + x chính là phần tử nhỏ nhất của nums2
        return min(nums2) - min(nums1)


sol = Solution()
for nums1, nums2, expected in examples:
    print(nums1, nums2, "->", sol.addedInteger(nums1, nums2))

[2, 6, 4] [9, 7, 5] -> 3
[10] [5] -> -5
[1, 1, 1, 1] [1, 1, 1, 1] -> 0


## 10. Vì sao thuật toán đúng

- **Thứ tự được giữ:** nếu `a <= b` thì `a + x <= b + x`. Vì vậy khi đa tập `nums1 + x` bằng đa tập `nums2`, phần tử nhỏ nhất tương ứng nhau: `min(nums1) + x = min(nums2)`. Từ đó `x = min(nums2) − min(nums1)`.
- **Duy nhất:** `x` được xác định bởi một đẳng thức tuyến tính nên chỉ có một giá trị.
- **Tương đương các công thức:** nếu `nums1 + x` bằng `nums2` thì `max` tương ứng, và tổng thỏa `sum(nums2) = sum(nums1) + n · x`. Cả ba công thức cho cùng `x`.
- **Không cần kiểm tra lại:** đề đảm bảo `x` tồn tại, nên công thức từ `min` là đủ. (Nếu `x` không tồn tại, công thức vẫn trả một số nhưng không đúng ý nghĩa — đề đã loại trường hợp đó.)

## 11. Phân tích Time / Space Complexity

Gọi `n` là độ dài mảng, `V = 2001` là số giá trị `x` có thể.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi `x` | `O(V · n)` | `O(n)` |
| 2. Ghép `nums1[0]` với từng `nums2[j]` | `O(n²)` | `O(n)` |
| 3. Hiệu hai `min` | `O(n)` | `O(1)` |
| 4. Hiệu hai `max` | `O(n)` | `O(1)` |
| 5. Hiệu hai tổng chia `n` | `O(n)` | `O(1)` |

Phải đọc mọi phần tử nên `O(n)` thời gian là tối ưu.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng. Mỗi test cũng kiểm chứng đáp án bằng cách **thực sự cộng `x` vào `nums1`** và so với `nums2` kiểu đa tập.

In [18]:
sol = Solution()
impls = {
    "brute": added_integer_bruteforce,
    "candidates": added_integer_candidates,
    "min": added_integer_min,
    "max": added_integer_max,
    "sum": added_integer_sum,
    "leetcode": sol.addedInteger,
}


def check_all(nums1, nums2, expected, label):
    for key, fn in impls.items():
        a, b = list(nums1), list(nums2)
        actual = fn(a, b)
        assert actual == expected, (label, key, nums1, nums2, actual, expected)
        assert a == nums1 and b == nums2, ("input bị sửa", key)
    assert Counter(v + expected for v in nums1) == Counter(nums2), "x kỳ vọng không thỏa đề"
    print("PASS", label)


for idx, (nums1, nums2, expected) in enumerate(examples, 1):
    check_all(nums1, nums2, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all([0], [0], 0, "một phần tử, x = 0")
check_all([0], [1000], 1000, "x lớn nhất dương")
check_all([1000], [0], -1000, "x nhỏ nhất (âm nhất)")
check_all([5, 5], [8, 8], 3, "các phần tử bằng nhau")
check_all([1, 2, 3], [3, 2, 1], 0, "cùng đa tập, thứ tự khác: x = 0")
check_all([1, 2, 3], [2, 3, 4], 1, "tăng 1")
check_all([3, 1, 2], [2, 3, 4], 1, "tăng 1, nums1 không theo thứ tự")
check_all([4, 4, 7], [1, 1, 4], -3, "x âm, có phần tử lặp")
check_all(list(range(100)), list(range(50, 150)), 50, "n = 100, tăng 50")
check_all(list(range(900, 1000)) + [], list(range(0, 100))[::-1], -900, "n = 100, x = -900, nums2 đảo ngược")

PASS một phần tử, x = 0
PASS x lớn nhất dương
PASS x nhỏ nhất (âm nhất)
PASS các phần tử bằng nhau
PASS cùng đa tập, thứ tự khác: x = 0
PASS tăng 1
PASS tăng 1, nums1 không theo thứ tự
PASS x âm, có phần tử lặp
PASS n = 100, tăng 50
PASS n = 100, x = -900, nums2 đảo ngược


Ngoài các ca chọn tay, mục 12.3 kiểm tra hàng nghìn input ngẫu nhiên hợp lệ (có `x` âm, `0`, dương và `nums2` bị xáo trộn).

### 12.2. Phản ví dụ cho các cách làm sai

Hai lỗi phổ biến: (a) **ghép theo thứ tự ban đầu** (`nums2[0] − nums1[0]`); (b) **đảo dấu** (`nums1 − nums2`).

In [20]:
def wrong_same_order(nums1, nums2):
    return nums2[0] - nums1[0]                           # SAI: hai mảng không cùng thứ tự


def wrong_sign(nums1, nums2):
    return min(nums1) - min(nums2)                       # SAI: đảo dấu


bad1 = ([2, 6, 4], [9, 7, 5])
bad2 = ([10], [5])
print(wrong_same_order(*bad1), "!=", added_integer_min(*bad1))
print(wrong_sign(*bad2), "!=", added_integer_min(*bad2))
assert wrong_same_order(*bad1) != added_integer_min(*bad1)
assert wrong_sign(*bad2) != added_integer_min(*bad2)

7 != 3
5 != -5


### 12.3. Đối chiếu ngẫu nhiên (có seed)

Sinh input **đúng ràng buộc**: chọn `nums1` ngẫu nhiên, chọn `x` sao cho mọi `nums1[i] + x` nằm trong `[0, 1000]`, rồi `nums2 = nums1 + x` **xáo trộn**. Mọi cách phải trả về đúng `x` đã dùng để sinh.

In [21]:
rng = random.Random(3131)
for trial in range(2000):
    n = rng.choice([1, 2, 3, 5, 10, 50, 100])
    hi = rng.choice([3, 10, 100, 1000])
    nums1 = [rng.randint(0, hi) for _ in range(n)]
    lo_x, hi_x = -min(nums1), 1000 - max(nums1)          # x sao cho mọi phần tử nums2 nằm trong [0, 1000]
    x = rng.randint(lo_x, hi_x)
    nums2 = [v + x for v in nums1]
    rng.shuffle(nums2)                                   # thứ tự khác nhau
    assert 1 <= len(nums1) == len(nums2) <= 100 and all(0 <= v <= 1000 for v in nums1 + nums2)
    for key, fn in impls.items():
        assert fn(list(nums1), list(nums2)) == x, (key, nums1, nums2, x)
print("PASS 2000 input ngẫu nhiên đúng ràng buộc: mọi cách trả về đúng x đã dùng để sinh dữ liệu")

PASS 2000 input ngẫu nhiên đúng ràng buộc: mọi cách trả về đúng x đã dùng để sinh dữ liệu


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Ghép các phần tử theo thứ tự ban đầu

In [22]:
def sai_ghep_theo_thu_tu(nums1, nums2):
    return nums2[0] - nums1[0]


print("sai:", sai_ghep_theo_thu_tu([2, 6, 4], [9, 7, 5]), "| đúng:", Solution().addedInteger([2, 6, 4], [9, 7, 5]))

sai: 7 | đúng: 3


**Vì sao sai:** đề chỉ nói hai mảng "bằng nhau" kiểu **đa tập**; `nums2` có thể bị xáo trộn so với `nums1 + x`. `nums2[0] − nums1[0] = 9 − 2 = 7` nhưng đáp án là `3`. **Cách sửa:** so sánh **phần tử nhỏ nhất** (hoặc lớn nhất), hoặc dùng tổng — những đại lượng **không phụ thuộc thứ tự**.

### Lỗi 2: Nhầm dấu

In [23]:
def sai_dau(nums1, nums2):
    return min(nums1) - min(nums2)           # SAI: phải là nums2 trừ nums1


print("sai:", sai_dau([10], [5]), "| đúng:", Solution().addedInteger([10], [5]))
print("sai:", sai_dau([2, 6, 4], [9, 7, 5]), "| đúng:", Solution().addedInteger([2, 6, 4], [9, 7, 5]))

sai: 5 | đúng: -5
sai: -3 | đúng: 3


**Vì sao sai:** ta cần `nums1 + x = nums2`, tức `x = nums2 − nums1`. Đảo vế cho `−x`. **Cách sửa:** luôn kiểm tra lại bằng cách **cộng `x` vào `nums1`** và so với `nums2` trên một ví dụ nhỏ.

### Lỗi 3: Dùng `/` khi chia tổng cho `n`

In [24]:
def sai_chia_thuc(nums1, nums2):
    return (sum(nums2) - sum(nums1)) / len(nums1)          # SAI: ra số thực


r = sai_chia_thuc([2, 6, 4], [9, 7, 5])
print("sai:", r, type(r).__name__, "| đúng:", Solution().addedInteger([2, 6, 4], [9, 7, 5]))
print("r == 3?", r == 3, "| isinstance(r, int)?", isinstance(r, int))

sai: 3.0 float | đúng: 3
r == 3? True | isinstance(r, int)? False


**Vì sao sai:** `/` luôn trả về **số thực** (`3.0`), trong khi đề yêu cầu số nguyên; LeetCode có thể báo sai kiểu dữ liệu. **Cách sửa:** `//` (chia nguyên), hoặc dùng `min`/`max`.

### Lỗi 4: Quên rằng `x` có thể âm hoặc bằng `0`

In [25]:
def sai_gia_su_duong(nums1, nums2):
    for x in range(1, 1001):                                # SAI: bỏ sót x = 0 và x < 0
        if Counter(v + x for v in nums1) == Counter(nums2):
            return x
    return None


print("sai:", sai_gia_su_duong([10], [5]), sai_gia_su_duong([1, 1, 1, 1], [1, 1, 1, 1]), "| đúng:", Solution().addedInteger([10], [5]), Solution().addedInteger([1, 1, 1, 1], [1, 1, 1, 1]))

sai: None None | đúng: -5 0


**Vì sao sai:** đề không nói `x > 0`; ví dụ 2 có `x = −5` và ví dụ 3 có `x = 0`. **Cách sửa:** vét cạn trên `range(-1000, 1001)`, hoặc dùng công thức hiệu không giả định dấu.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cộng **cùng một số** vào mọi phần tử" ⇒ mọi **đại lượng thống kê tuyến tính** đều dịch chuyển đúng `x`: `min`, `max`, trung bình, trung vị, tổng/`n`. Chọn đại lượng rẻ nhất (ở đây `min`).
- "Bằng nhau **không cần thứ tự**" ⇒ đừng so theo vị trí; dùng `min/max`, `sum`, `sorted`, hoặc `Counter`.
- Kiểm tra bằng **thực sự cộng `x`** vào `nums1` và so sánh đa tập là cách tự kiểm tra chắc chắn.
- Chia nguyên `//` khi kết quả phải là số nguyên; nhớ `//` làm tròn **xuống** với số âm.
- Nếu đề **không** đảm bảo `x` tồn tại: phải kiểm chứng thêm (bằng `sorted` hoặc `Counter`).

## 15. Biến thể của bài

1. **Find the Integer Added to Array II (3132):** được **xóa hai phần tử** khỏi `nums1` rồi mới cộng `x`; cần thử ứng viên và kiểm tra.
2. **Kiểm tra có tồn tại `x`** hay không (trả `None` nếu không).
3. **Nhân thay vì cộng** (`nums2 = nums1 · k`).
4. **`nums2 = nums1 + x` theo từng vị trí** (đúng thứ tự): lúc này `x = nums2[0] − nums1[0]` đúng!
5. **Minimum Operations to Make Array Equal (1551)**, **Minimum Moves to Equal Array Elements (453)**.

In [26]:
def exists_added_integer(nums1, nums2):
    if len(nums1) != len(nums2):
        return None
    x = min(nums2) - min(nums1)
    return x if Counter(v + x for v in nums1) == Counter(nums2) else None


def added_integer_same_order(nums1, nums2):
    return nums2[0] - nums1[0]


print(exists_added_integer([2, 6, 4], [9, 7, 5]), exists_added_integer([1, 2], [1, 4]))   # 3 None
print(added_integer_same_order([2, 6, 4], [5, 9, 7]))                                     # 3 (đúng thứ tự)
assert exists_added_integer([2, 6, 4], [9, 7, 5]) == 3
assert exists_added_integer([1, 2], [1, 4]) is None
assert added_integer_same_order([2, 6, 4], [5, 9, 7]) == 3

3 None
3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi `x` | `Counter(nums1 + x) == Counter(nums2)` | `O(V · n)` | `O(n)` | làm chuẩn kiểm chứng |
| 2. Ứng viên `nums2[j] − nums1[0]` | kiểm tra từng ứng viên | `O(n²)` | `O(n)` | khi không tin vào công thức |
| 3. Hiệu hai `min` | `min(nums2) − min(nums1)` | `O(n)` | `O(1)` | **nên nộp** |
| 4. Hiệu hai `max` | `max(nums2) − max(nums1)` | `O(n)` | `O(1)` | tương đương cách 3 |
| 5. Hiệu hai tổng chia `n` | `(sum2 − sum1) // n` | `O(n)` | `O(1)` | khi muốn dùng mọi phần tử |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sorted([3, 1, 2])`, `Counter([1, 1, 2]) == Counter([2, 1, 1])`, `-9 // 3`, `Solution().addedInteger([1, 5], [4, 8])`, `Solution().addedInteger([8, 3], [1, 6])`.

**Bài 2 (tính tay).** Với `nums1 = [7, 3, 10]` và `nums2 = [5, 1, 8]`: tính `x` bằng ba cách (`min`, `max`, tổng) và kiểm tra bằng cách cộng `x` vào `nums1`.

**Bài 3 (điền TODO).** Hoàn thành `added_integer_todo(nums1, nums2)` theo Cách 5 (hiệu tổng chia `n`).

In [27]:
def added_integer_todo(nums1, nums2):
    n = len(nums1)
    # TODO 1: tính hiệu hai tổng: sum(nums2) - sum(nums1)
    # TODO 2: chia nguyên cho n (dùng //, không dùng /)
    # TODO 3: trả về kết quả
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `added_integer_sorted(nums1, nums2)`: sắp xếp cả hai mảng rồi trả về `sorted(nums2)[0] − sorted(nums1)[0]`; sau đó **kiểm tra** bằng một vòng `all(...)` rằng mọi cặp `(a, b)` cùng vị trí đều có `b − a == x`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `added_integer_or_none(nums1, nums2)` trả về `x` nếu tồn tại, ngược lại `None` (kể cả khi độ dài khác nhau). (b) Viết `multiplied_integer(nums1, nums2)` cho bài **nhân** (`nums1[i] · k` rồi so đa tập với `nums2`; các phần tử dương): trả về `k` (số nguyên dương) hoặc `None`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 2, 3]`, `True`, `-3`, `3` (min: `4 − 1`), `-2` (`min(nums2) = 1`, `min(nums1) = 3`: `1 − 3 = −2`). Bài 3: `(sum(nums2) - sum(nums1)) // n`.
- **Mức 2 (vừa):** Bài 2: `min`: `1 − 3 = −2`; `max`: `8 − 10 = −2`; tổng: `(14 − 20) // 3 = −2`; kiểm tra: `[7 − 2, 3 − 2, 10 − 2] = [5, 1, 8]`, đúng bằng `nums2`. Bài 4: sắp xếp rồi `zip`.
- **Mức 3 (sâu):** Bài 5(a): tính `x = min(nums2) − min(nums1)` rồi kiểm tra `Counter(v + x for v in nums1) == Counter(nums2)`; nhớ kiểm tra độ dài trước. 5(b): `k = min(nums2) // min(nums1)` nếu chia hết (chú ý `min(nums1) > 0`), rồi kiểm tra đa tập.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert sorted([3, 1, 2]) == [1, 2, 3]
assert (Counter([1, 1, 2]) == Counter([2, 1, 1])) is True
assert -9 // 3 == -3
assert Solution().addedInteger([1, 5], [4, 8]) == 3
assert Solution().addedInteger([8, 3], [1, 6]) == -2

# Bài 2 (dùng ví dụ nums1 = [7, 3, 10], nums2 = [5, 1, 8])
n1, n2 = [7, 3, 10], [5, 1, 8]
assert min(n2) - min(n1) == -2
assert max(n2) - max(n1) == -2
assert (sum(n2) - sum(n1)) // len(n1) == -2
assert Counter(v - 2 for v in n1) == Counter(n2)


# Bài 3
def added_integer_todo_ans(nums1, nums2):
    n = len(nums1)
    return (sum(nums2) - sum(nums1)) // n


for nums1, nums2, expected in examples:
    assert added_integer_todo_ans(nums1, nums2) == expected


# Bài 4
def added_integer_sorted(nums1, nums2):
    s1, s2 = sorted(nums1), sorted(nums2)
    x = s2[0] - s1[0]
    assert all(b - a == x for a, b in zip(s1, s2))
    return x


for nums1, nums2, expected in examples:
    assert added_integer_sorted(nums1, nums2) == expected


# Bài 5(a)
def added_integer_or_none(nums1, nums2):
    if len(nums1) != len(nums2) or not nums1:
        return None
    x = min(nums2) - min(nums1)
    return x if Counter(v + x for v in nums1) == Counter(nums2) else None


assert added_integer_or_none([2, 6, 4], [9, 7, 5]) == 3
assert added_integer_or_none([1, 2], [1, 4]) is None
assert added_integer_or_none([1], [1, 2]) is None


# Bài 5(b)
def multiplied_integer(nums1, nums2):
    if len(nums1) != len(nums2) or not nums1 or min(nums1) <= 0:
        return None
    if min(nums2) % min(nums1) != 0:
        return None
    k = min(nums2) // min(nums1)
    return k if k > 0 and Counter(v * k for v in nums1) == Counter(nums2) else None


assert multiplied_integer([1, 2, 3], [3, 6, 9]) == 3
assert multiplied_integer([2, 4], [8, 4]) == 2
assert multiplied_integer([1, 2], [2, 5]) is None
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (cộng cùng `x` vào mọi phần tử ⇒ `x = min(nums2) − min(nums1)`)

```python
from typing import List

class Solution:
    def addedInteger(self, nums1: List[int], nums2: List[int]) -> int:
        return min(nums2) - min(nums1)
```

**Checklist**

- [ ] Dùng đại lượng **không phụ thuộc thứ tự** (`min`, `max`, tổng), không ghép theo vị trí?
- [ ] Dấu đúng: `x = nums2 − nums1` (kiểm tra bằng cách cộng `x` vào `nums1`)?
- [ ] Nếu dùng tổng: chia nguyên `//` cho `n`, không dùng `/`?
- [ ] Chấp nhận `x` âm hoặc `0`?
- [ ] Đã thử `[10] → [5]` (âm), `[1,1,1,1] → [1,1,1,1]` (không), và mảng có thứ tự khác nhau?
- [ ] Skeleton đúng `class Solution` / `def addedInteger(self, nums1: List[int], nums2: List[int]) -> int`?